# Phase 2: Advanced Modelling & MLOps

## Day 5: MLflow Initialisation

Objectives:
- Install and configure a local MLflow tracking server.
- Set up the experiment namespace for the project.
- Run a verification test to confirm the MLflow connection.

## Step 2: Configure MLflow in the Notebook

> **Prerequisites:** Ensure the MLflow UI server is running in a separate terminal via `mlflow ui`.

In [ ]:
import mlflow

# Configure the local tracking URI to point to your running server
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Set up the experiment namespace for the project
experiment_name = "RMIT_Fraud_Detection_Sprint"
mlflow.set_experiment(experiment_name)

print(f"Connected to MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Active Experiment: {experiment_name}")

## Step 3: Run a Quick Verification Test

Log a dummy run to verify the end-to-end connection to the MLflow tracking server.

In [ ]:
with mlflow.start_run(run_name="Day_5_Initialization_Test"):
    # Log a test parameter
    mlflow.log_param("test_parameter", "successful")

    # Log a test metric
    mlflow.log_metric("test_metric", 1.0)

print("Test run logged successfully. Check your MLflow UI at http://127.0.0.1:5000")

---
## Data Setup
Re-load and preprocess the data so this notebook is self-contained.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('../data/raw/creditcard.csv')

# Apply RobustScaler to Amount and Time
rob_scaler = RobustScaler()
df['scaled_amount'] = rob_scaler.fit_transform(df['Amount'].values.reshape(-1, 1))
df['scaled_time']   = rob_scaler.fit_transform(df['Time'].values.reshape(-1, 1))
df.drop(['Time', 'Amount'], axis=1, inplace=True)

# Reorder so scaled features are at the front
scaled_amount = df.pop('scaled_amount')
scaled_time   = df.pop('scaled_time')
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time',   scaled_time)

# Stratified 80/20 split
X = df.drop('Class', axis=1)
y = df['Class']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Training set: {X_train.shape} | Test set: {X_test.shape}')

---
# Day 6: SMOTE Integration

Objectives:
- Apply SMOTE **exclusively** to the training set to prevent data leakage.
- Verify the synthetic samples produce a perfectly balanced training set.

## Step 1: Apply SMOTE to the Training Data

In [ ]:
from imblearn.over_sampling import SMOTE

# Instantiate SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE strictly to the training data — never touch the test set!
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print('SMOTE applied successfully.')

## Step 2: Verify Synthetic Sample Quality

In [ ]:
print('Before SMOTE class distribution:')
print(y_train.value_counts())

print('\nAfter SMOTE class distribution:')
print(pd.Series(y_train_smote).value_counts())

---
# Day 7: Advanced Model Training & MLflow Logging

Objectives:
- Train a Random Forest classifier on the SMOTE-balanced training data.
- Log hyperparameters, F1-score, and AUPRC to MLflow.
- Store the serialized model artifact in MLflow.

## Step 1: Train the Random Forest and Log to MLflow

> **Prerequisite:** Ensure `mlflow ui` is still running in a separate terminal.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve, auc, f1_score
import mlflow
import mlflow.sklearn

# Ensure we are still pointing at the local tracking server
mlflow.set_tracking_uri('http://127.0.0.1:5000')
mlflow.set_experiment('RMIT_Fraud_Detection_Sprint')

# Set up hyperparameters
params = {
    'n_estimators': 100,
    'max_depth': 10,
    'random_state': 42
}

# Start the MLflow run
with mlflow.start_run(run_name='RandomForest_SMOTE'):

    # Train on SMOTE-balanced training data
    rf_model = RandomForestClassifier(**params)
    rf_model.fit(X_train_smote, y_train_smote)

    # Evaluate on the untouched test set
    y_pred       = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

    # Calculate metrics
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    auprc_score = auc(recall, precision)
    f1 = f1_score(y_test, y_pred)

    # Log hyperparameters and metrics to MLflow
    mlflow.log_params(params)
    mlflow.log_metric('auprc', auprc_score)
    mlflow.log_metric('f1_score', f1)

    # Log the model artifact
    mlflow.sklearn.log_model(rf_model, 'random_forest_model')

    print(f'Run completed.  AUPRC: {auprc_score:.4f} | F1: {f1:.4f}')

---
# Day 8: Model Selection & Export

Objectives:
- Select the best-performing model (confirmed via MLflow UI).
- Serialize it to `models/model.pkl` using joblib for use in the Phase 3 API.

## Step 2: Serialize the Model

In [ ]:
import joblib
import os

# Ensure the models directory exists
os.makedirs('../models', exist_ok=True)

# Export the best trained model
joblib.dump(rf_model, '../models/model.pkl')

print('Model successfully exported to models/model.pkl')

In [ ]:
# Also export the fitted scaler — the API needs this to preprocess raw inputs
joblib.dump(rob_scaler, '../models/scaler.pkl')
print('Scaler successfully exported to models/scaler.pkl')